In [1]:
from poke_env import RandomPlayer
from poke_env.data import GenData
from poke_env.player import Player
from poke_env.battle.pokemon import Pokemon
from poke_env.battle.move import Move

In [2]:
x = Move('selfdestruct',2)

In [3]:
x.id

'selfdestruct'

In [4]:
x.entry

{'accuracy': 100,
 'basePower': 200,
 'category': 'Physical',
 'contestType': 'Beautiful',
 'flags': {'metronome': 1,
  'mirror': 1,
  'noparentalbond': 1,
  'nosketch': 1,
  'protect': 1},
 'name': 'Self-Destruct',
 'num': 120,
 'pp': 5,
 'priority': 0,
 'secondary': None,
 'selfdestruct': 'always',
 'target': 'allAdjacent',
 'type': 'Normal'}

In [5]:
z = Move('dragonrage',1)
z.entry

{'accuracy': 100,
 'basePower': 1,
 'category': 'Special',
 'contestType': 'Cool',
 'damage': 40,
 'flags': {'metronome': 1, 'mirror': 1, 'protect': 1},
 'isNonstandard': None,
 'name': 'Dragon Rage',
 'num': 82,
 'pp': 10,
 'priority': 0,
 'secondary': None,
 'target': 'normal',
 'type': 'Dragon'}

In [6]:
x.entry

{'accuracy': 100,
 'basePower': 200,
 'category': 'Physical',
 'contestType': 'Beautiful',
 'flags': {'metronome': 1,
  'mirror': 1,
  'noparentalbond': 1,
  'nosketch': 1,
  'protect': 1},
 'name': 'Self-Destruct',
 'num': 120,
 'pp': 5,
 'priority': 0,
 'secondary': None,
 'selfdestruct': 'always',
 'target': 'allAdjacent',
 'type': 'Normal'}

In [7]:
y = Move('thunderbolt',1)
y.entry

{'accuracy': 100,
 'basePower': 95,
 'category': 'Special',
 'contestType': 'Cool',
 'flags': {'metronome': 1, 'mirror': 1, 'protect': 1},
 'name': 'Thunderbolt',
 'num': 85,
 'pp': 15,
 'priority': 0,
 'secondary': {'chance': 10, 'status': 'par'},
 'target': 'normal',
 'type': 'Electric'}

In [8]:
x._moves_dict['painsplit']

{'accuracy': 100,
 'basePower': 0,
 'category': 'Status',
 'contestType': 'Clever',
 'flags': {'allyanim': 1, 'metronome': 1, 'mirror': 1, 'protect': 1},
 'name': 'Pain Split',
 'num': 220,
 'onHit': 'onHit',
 'pp': 20,
 'priority': 0,
 'secondary': None,
 'target': 'normal',
 'type': 'Normal',
 'zMove': {'boost': {'def': 1}}}

In [9]:
x = Pokemon(gen=2,species='snorlax')

In [10]:
x.item

'unknown_item'

In [11]:
x._update_current_stats({'spd':100})

{'spd': 100}


In [12]:
x.stats_current

{'hp': None, 'atk': None, 'def': None, 'spa': None, 'spd': 100, 'spe': None}

In [13]:
x._gen

2

In [18]:
class Pokebot_Gen1(Player):            

    def choose_move(self, battle):

        # if battle.turn>2 and battle.turn<6:
            # print(battle.turn)
            # print(f'Current Stats:{battle.active_pokemon.stats_current}')
        if battle.turn == 3:
            print(battle.player_role)
            print(battle.observations)
        return self.choose_random_move(battle)

In [15]:
teams = """
Mewtwo  
Ability: No Ability  
- Thunder Wave 
- Recover
- Psychic 

Mew  
Ability: No Ability  
- Thunder Wave 
- Psychic 
"""

In [ ]:
random_player = Pokebot_Gen1(battle_format='gen1ubers',team=teams)
second_player = RandomPlayer(battle_format='gen1ubers',team=teams)


In [20]:
await random_player.battle_against(second_player, n_battles=1)

In [1]:
import logging
from poke_env.battle import Battle, Move, Pokemon
from poke_env.battle.side_condition import SideCondition
from poke_env.stats import compute_raw_stats_dvs
from typing import List
from poke_env.teambuilder import Teambuilder
from poke_env.player import Player
from poke_env.player.battle_order import ForfeitBattleOrder
attack_team = """
Venusaur  
Ability: No Ability  
- Stun Spore 
- Swords Dance
- Rest
- Growl

Jolteon  
Ability: No Ability  
- Thunder Wave 
- Agility
- Growl
- Rest  
"""
defense_team = """
Pikachu  
Ability: No Ability  
Level: 10   
- Seismic Toss  
- Quick Attack
- Thunder Wave

Magnemite  
Ability: No Ability  
- Thunder Wave  
"""

class Attack_Player(Player):
    def __init__(self, account_configuration = None, *, avatar = None, battle_format = "gen1ou", log_level = None, max_concurrent_battles = 1, accept_open_team_sheet = False, save_replays = False, server_configuration = ..., start_timer_on_battle_start = False, start_listening = True, open_timeout = 10, ping_interval = 20, ping_timeout = 20, team = attack_team):
        super().__init__(account_configuration, avatar=avatar, battle_format=battle_format, log_level=log_level, max_concurrent_battles=max_concurrent_battles, accept_open_team_sheet=accept_open_team_sheet, save_replays=save_replays, server_configuration=server_configuration, start_timer_on_battle_start=start_timer_on_battle_start, start_listening=start_listening, open_timeout=open_timeout, ping_interval=ping_interval, ping_timeout=ping_timeout, team=team)

        self.player_counter = 0

    def choose_move(self, battle):
        obs = battle.observations
        last_obs_key = max(obs)
        last_obs = obs[last_obs_key]
        events = last_obs.events
        # return ForfeitBattleOrder()
        for e in events:
            event = e[:]


            if event[1] == "move":
                failed = False


                for move_failed_suffix in ["[miss]", "[still]", "[notarget]"]:
                    if event[-1] == move_failed_suffix:
                        event = event[:-1]
                        failed = True
                                
                pokemon, move, presumed_target = event[2:5]
                mover = pokemon[0:2]
                if mover == battle.player_role:
                    if failed and self.player_counter < 20:
                        break
                    elif failed and self.player_counter >=20:
                        continue
                    else:
                        self.player_counter+=1


        if self.player_counter == 0:
            
            m = battle.available_moves[0]
            return self.create_order(m)
        
        elif self.player_counter == 1:
            m = battle.available_moves[1]
            return self.create_order(m)
        elif self.player_counter == 2:
            m = battle.available_moves[2]
            return self.create_order(m)
        else:
            return ForfeitBattleOrder()


In [2]:
from poke_env import RandomPlayer
from poke_env.data import GenData
from poke_env.player import Player
from poke_env.battle.pokemon import Pokemon
from poke_env.battle.move import Move


In [3]:
ap = Attack_Player(battle_format='gen1ou',team=attack_team)
# dp = RandomPlayer(battle_format='gen1ou',team=defense_team)

2025-09-29 15:51:42,754 - Attack_Player 1 - ERROR - 'ellipsis' object has no attribute 'websocket_url'
Traceback (most recent call last):
  File "/home/jacobkim/Desktop/VS_Code/poke-env/src/poke_env/ps_client/ps_client.py", line 217, in listen
    self.websocket_url,
    ^^^^^^^^^^^^^^^^^^
  File "/home/jacobkim/Desktop/VS_Code/poke-env/src/poke_env/ps_client/ps_client.py", line 364, in websocket_url
    return self.server_configuration.websocket_url
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AttributeError: 'ellipsis' object has no attribute 'websocket_url'


In [ ]:
await random_player.battle_against(second_player, n_battles=1)

In [18]:
x = {1:2,2:4}
max(x)

2